# ResNet

ResNet 是 2015 年 Kaiming He 等人在微软亚洲研究院提出的工作。它主要想解决的问题是：当神经网络变得越来越深时，模型反而可能更难训练，效果不一定更好。

当时的深度神经网络随着网络层数增加，主要会面临几个问题：

1. 梯度爆炸 / 梯度消失

由于反向传播依赖链式法则，层数越深，梯度在一层层传回去的过程中就越容易不断放大或不断缩小，最终出现梯度爆炸或梯度消失。当然，这个问题可以通过 normalization、合理初始化、激活函数设计等方法缓解。

2. 网络退化

网络退化（degradation）指的是：当网络继续加深时，训练误差反而变高。这个现象不完全是过拟合，因为过拟合通常表现为训练集好、测试集差；而网络退化中，训练集本身也可能变差。

直观上，如果一个更深的网络真的不需要多出来的层，那么这些层只要学成 identity mapping，理论上就不会比浅层网络差。但问题是，在普通的非线性网络中，让一堆卷积、BN、ReLU 精确学到 identity mapping 并不容易。

## Skip Connection

ResNet 的解决方式是 skip connection，也叫 shortcut connection，如下图：

![Residual learning block](../../figs/resnet.png)

普通网络希望一个模块直接学习目标映射：

$$
H(x)
$$

而 ResNet 不让模块直接学习 $H(x)$，而是让它学习一个残差项：

$$
F(x)=H(x)-x
$$

最后模块输出为：

$$
y = F(x) + x
$$

这样一来，如果这个模块暂时不需要学习额外变化，只需要让 $F(x) \approx 0$，输出就接近：

$$
y \approx x
$$

也就是说，相比让一堆层直接学出 identity mapping，让它们学出接近 0 的 residual branch 通常更容易。这就是 ResNet 缓解网络退化问题的核心直觉。

同时，

假设：

$$
y = F(x) + x
$$

那么反向传播时：

$$
\frac{\partial L}{\partial x}
= \frac{\partial L}{\partial y}\left(\frac{\partial F(x)}{\partial x}+I\right)
$$

这里的 $I$ 来自 skip connection。它表示梯度除了经过 residual branch 中的卷积、BN、ReLU 等操作之外，还可以沿着 identity shortcut 直接往前传。这样即使 residual branch 的梯度传播不够理想，梯度也仍然有机会沿着 shortcut 往前传，从而缓解深层网络中梯度难以传递的问题。

In [3]:
# ResNet 

import torch
import torch.nn as nn
import torch.nn.functional as F

class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion*planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion*planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion*planes)
            ) # let shape of x equal to shape of out, so that we can add them together

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out

In [4]:
x = torch.randn(1, 3, 32, 32)
model = BasicBlock(3, 16)
y = model(x)
print(y.shape)

torch.Size([1, 16, 32, 32])
